In [1]:
import gymnasium as gym
import numpy as np

In [2]:
# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8


In [3]:
# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------
def policy_evaluation(env, policy, gamma=0.99, theta=1e-8):
    V = np.zeros(env.observation_space.n)
    iteration = 0

    while True:
        delta = 0

        for s in range(env.observation_space.n):
            v = V[s]
            new_v = 0

            for a, action_prob in enumerate(policy[s]):
                for prob, next_state, reward, done in env.P[s][a]:
                    new_v += action_prob * prob * (
                        reward + gamma * V[next_state] * (not done)
                    )

            V[s] = new_v
            delta = max(delta, abs(v - V[s]))

        iteration += 1

        if delta < theta:
            break

    return V, iteration


In [4]:
# -------------------------------------------------
# Generate Value Function Before Policy Improvement
# -------------------------------------------------

policy = np.ones((n_states, n_actions)) / n_actions

V, iteration = policy_evaluation(env, policy, gamma, theta)

print("Value Function Before Policy Improvement:")
print(np.round(V.reshape(4, 4), 4))
print("Number of iterations:", iteration)

Value Function Before Policy Improvement:
[[0.0124 0.0104 0.0193 0.0095]
 [0.0148 0.     0.0389 0.    ]
 [0.0326 0.0843 0.1378 0.    ]
 [0.     0.1703 0.4336 0.    ]]
Number of iterations: 54


In [14]:
# -------------------------------------------------
# Policy Improvement
# -------------------------------------------------
def policy_improvement(env, V, policy, gamma=0.99):
    policy_stable = True
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    for s in range(n_states):
        old_action = np.argmax(policy[s])
        action_values = np.zeros(n_actions)

        for a in range(n_actions):
            for prob, next_state, reward, done in env.P[s][a]:
                action_values[a] += prob * (
                    reward + gamma * V[next_state] * (not done)
                )

        best_action = np.argmax(action_values)

        policy[s] = np.eye(n_actions)[best_action]

        if old_action != best_action:
            policy_stable = False

    return policy, policy_stable

In [15]:

# -------------------------------------------------
# Generate Policy Improvement
# -------------------------------------------------

policy = np.ones((n_states, n_actions)) / n_actions

V, iteration = policy_evaluation(env, policy, gamma, theta)

policy, policy_stable = policy_improvement(env, V, policy, gamma)

print("Policy After Improvement:")
print_policy(policy)
print("Policy Stable:", policy_stable)

Policy After Improvement:

Optimal Policy:
[['←' '↑' '←' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]
Policy Stable: False


In [16]:

# -------------------------------------------------
# Policy Iteration
# -------------------------------------------------

def policy_iteration(env, gamma=0.99, theta=1e-8):
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    policy = np.ones((n_states, n_actions)) / n_actions

    while True:
        V, iteration = policy_evaluation(env, policy, gamma, theta)

        policy, policy_stable = policy_improvement(
            env, V, policy, gamma
        )

        if policy_stable:
            break

    return policy, V



In [17]:

# -------------------------------------------------
# Display Functions
# -------------------------------------------------

def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))


def print_policy(policy):
    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }

    best_actions = np.argmax(policy, axis=1)
    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)


    print("\nOptimal Policy:")
    print(policy_grid)



In [18]:
# -------------------------------------------------
# Run Policy Iteration
# -------------------------------------------------

optimal_policy, optimal_value_function = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("Name: VIDHIYA LAKSHMI S")
print("Register Number: 212223230238")
print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()

Name: VIDHIYA LAKSHMI S
Register Number: 212223230238

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Optimal Policy:
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]
